<a href="https://colab.research.google.com/github/kdkim2000/RAG2026/blob/main/%5B%EC%8B%A4%EC%8A%B5%5D_5_PDF_%EB%82%B4%EC%9A%A9_%EA%B8%B0%EB%B0%98_%EC%A7%88%EC%9D%98%EC%9D%91%EB%8B%B5_%EC%96%B4%ED%94%8C%EB%A6%AC%EC%BC%80%EC%9D%B4%EC%85%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [실습] PDF 내용 기반 질의응답 어플리케이션

이번에는 PDF의 내용을 이용해 질의응답을 수행해 보겠습니다.

### 라이브러리 설치  

랭체인 관련 라이브러리와 벡터 데이터베이스 라이브러리를 설치합니다.   

In [ ]:
!pip install openai langchain langchain-openai langchain-community chromadb tiktoken langchain_chroma pymupdf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

OpenAI API 키 확인


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.6", reasoning_effort='low')

## Indexing : 데이터 불러오기

PyMuPDFLoader를 이용해 목표 문서를 불러옵니다.    
실습 시트에 포함된 교재 PDF 파일을 업로드해 주세요.

In [ ]:
# 현재 위치의 모든 pdf 불러오기
from glob import glob
import os

pdfs = glob(os.path.join('./', '*.pdf'))
pdfs

['./교재_0803.pdf']

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

documents = []

for path_material in pdfs:
    print(path_material)
    loader = PyMuPDFLoader(path_material)
    # 페이지별로 저장
    pages = loader.load()
    documents.extend(pages)
    print("# Number of pages:", len(pages))

print("# Total pages:", len(documents))

/tmp/ipykernel_1824/4215178708.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


./교재_0803.pdf
# Number of pages: 82
# Total pages: 82


PDF Loader로 불러온 데이터는 페이지 단위로 저장됩니다.    

In [ ]:
for i in range(10,15):
    print(documents[i].page_content)
    print('----------')

LangChain 1.0 (2025.10)
기존LangChain 라이브러리의분리및개편
- langchain : LLM Agent 개발을위한모듈위주
- langchain-classic: 기존의다양한도구들
기존코드가돌아가지않는경우
- langchain_classic으로수정하면대부분해결됨
- https://docs.langchain.com/oss/python/migrate/langchain-v1
11
----------
Chat Models in LangChain
12
----------
Runnable과Invoke
LangChain의구성요소는Runnable 클래스
- Runnable 계열의클래스는invoke()를통해동기실행, ainvoke()로비동기실행
- batch()로병렬실행, abatch()로병렬비동기실행
Init_chat_model()을통한API 통합연결
- Model_provider, model_name 전달
- SystemMessage, HumanMessage, AIMessage 형태로메시지클래스입력
13
----------
[실습] LangChain 기본구조
기본프롬프트구성방법이해하기
다양한LLM 모델과LangChain 연결하기
병렬실행을위한batch, 순차적출력을위한stream 사용하기
모델의프롬프트템플릿과LLM 연동하기
14
----------
LangChain 자동화및챗봇만들기
15
----------


각각의 Document를 하나로 합쳐, 하나의 큰 Document를 만들고 청킹을 수행하겠습니다.

In [ ]:
from langchain_core.documents import Document
# Document 클래스 만들기

corpus = Document(page_content='', metadata={'source': ', '.join(pdfs)})
for page in documents:
    corpus.page_content += page.page_content + '\n'

corpus.page_content = corpus.page_content.replace('\n\n','\n')
len(corpus.page_content)

14592

Text Splitter를 이용해 청크로 분리합니다.   
이번에는 토큰 단위로 분리해 보겠습니다.    
TextSplitter의 `.from_tiktoken_encoder`를 이용합니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4o-mini",
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = token_splitter.split_documents([corpus])
print(len(chunks))

11


전체 데이터를 Chroma에 저장합니다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')
Chroma().delete_collection()
db = Chroma.from_documents(chunks, embeddings)


retriever = db.as_retriever(search_kwargs={"k": 5})
# Top K search 옵션 정하기


# filter 옵션을 통해 특정 메타데이터를 가진 벡터만 검색 가능
# Ex) author가 Hyungho Byun인 Document만 검색
# retriever = db.as_retriever(search_kwargs={"k": 5,"filter":{'author':'Hyungho Byun'}})

In [ ]:
# Query 검색
unique_docs = retriever.invoke("LangChain의 장점은?")

unique_docs

[Document(id='02c9287c-1a73-404f-aa70-646b486efe5d', metadata={'source': './교재_0803.pdf'}, page_content='- 질문이입력되면, 질문의임베딩과청크들의임베딩유사도계산\n- Top K Chunks를Return\n\uf071청크검색후, Task 에따른활용\n\uf0e0RAG : LLM 프롬프트에추가하여답변생성\n\uf0e0Recommendation : 해당상품혹은문서전달\n34\nSearching: 대표적Vector Database\n\uf071Vector 검색전용DB\n- Pinecone : 클라우드기반의유료서비스\n- Milvus, Qdrant, Chroma, Weaviate : Online/Self Host 사용지원\n\uf071기존DB(Elastic Search, OpenSearch) 등의벡터검색지원\n- 벡터수가많지않고, 기존DB를운영한다면기존DB에서활용권장\n35\nSearching: 대표적Vector Database\n\uf071Vector Database의Metric Types\n- Euclidean Distance (L2 Distance)\n－벡터간의직선거리\n- Cosine Distance (임베딩이Normalized인경우)\n－1- Cosine Similarity\n- MMR (Maximum Marginal Relevance Search)\n－유사성과다양성의균형을고려한방법\n－Alpha * 쿼리와의유사성– (1-Alpha) * (Context 중가장가까운청크와의유사성)\n순으로검색\n36\n[실습] 벡터데이터베이스기반RAG 어플리케이션\n\uf071WebBaseLoader()를통한웹페이지내용기반의RAG\n\uf071ChromaDB를이용하여뉴스기사청크저장및검색\n37\n[실습] PDF 내용기반질의응답어플리케이션\n\uf071PDF 파일의내용을기반으로연속적질의응답수행하기\n\uf071요약알고리즘의stuff, map-reduce, refine 과정수행하기\n38\n2025년

## [실습] PDF 질의응답 체인 만들기    

Prompt와 Chain을 구성하여, 강의 자료에 대한 질의응답을 수행하는 RAG 체인을 만들고 실행하세요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    당신은 주어진 문맥을 바탕으로 질문에 답변하는 친절한 AI 어시스턴트입니다.
    주어진 정보를 벗어나서 답변하지 마세요.
    모르는 내용은 모른다고 솔직하게 답변하세요.\n\n{context}"""),
    ("human", "{question}")
])

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} # retriever가 context를 제공하고, question은 그대로 전달
    | prompt # prompt 템플릿에 context와 question을 전달
    | llm # LLM을 호출하여 답변 생성
    | StrOutputParser() # LLM의 출력을 문자열로 파싱
)

In [ ]:
# 테스트
rag_chain.invoke("LangChain의 장점은?")

'LangChain의 주요 장점은 다음과 같습니다.\n\n- **높은 추상화**를 제공해 LLM 애플리케이션을 쉽게 구현할 수 있음\n- 모델, 프롬프트, 체인, 데이터베이스, 메모리 등을 **모듈식으로 연결**할 수 있음\n- **툴 사용과 에이전트 개발**을 지원함\n- 다양한 LLM 제공자와 오픈 모델을 별도 라이브러리로 연결할 수 있음\n- 공식 문서와 예제 코드가 풍부해 **바이브 코딩으로도 구현하기 쉬움**\n\n즉, LangChain은 LLM 애플리케이션 개발에 필요한 기능을 모아 놓은 **“LLM Application Toolbox”**라고 할 수 있습니다.'

Top-K 기반의 RAG도 문서의 내용을 바탕으로 답변하지만,    
특정 문제에 대해서는 단일 검색이 아닌 전체 문서를 확인해야 하는 경우가 존재합니다.   

Chunking을 활용하여, PDF 파일을 요약해 보겠습니다.

## 요약(Summarization)   

### 1. Stuff : 전체 문서를 다 넣고 요약하기

가장 간단한 요약 방법입니다.   
문서의 길이가 Context 길이보다 큰 경우에는 실행이 어렵습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

stuff_prompt = ChatPromptTemplate([
    ("system", """당신은 전문적인 문서 요약 전문가입니다.
주어진 문서를 읽고 핵심 내용을 체계적으로 요약해주세요.

요약 가이드라인:
1. 문서의 주요 주제와 목적을 먼저 파악하세요
2. 핵심 내용을 구조화하여 정리하세요
3. 중요한 수치나 데이터는 포함하세요
4. 전문 용어는 그대로 사용하되, 맥락을 명확히 하세요"""),
    ("human", """다음 문서를 요약해주세요.

---
{text}
---

위 문서의 핵심 내용을 체계적으로 요약해주세요.""")
])

# Stuff 체인 구성
stuff_chain = stuff_prompt | llm | StrOutputParser()


In [ ]:
import time
# Stuff 방식 실행
print("Stuff 방식으로 요약 중...")
start_time = time.time()

stuff_summary = stuff_chain.invoke({"text": corpus.page_content})

elapsed_time = time.time() - start_time
print(f"완료! (소요 시간: {elapsed_time:.2f}초)\n")
print("="*60)
print("[Stuff 요약 결과]")
print("="*60)
print(stuff_summary)

Stuff 방식으로 요약 중...
완료! (소요 시간: 40.07초)

[Stuff 요약 결과]
# 문서 요약: 2026 삼성SDS AI 교육 ― RAG 개발 심화

## 1. 교육 개요와 목적

본 문서는 **LangChain을 활용한 LLM 애플리케이션 및 고급 RAG(Retrieval Augmented Generation) 시스템 개발**을 다루는 2일 과정의 교육자료다.

### 과정 목표
- LLM의 기본 개념과 대표 활용 사례 이해
- LangChain의 주요 컴포넌트와 개발 방식 습득
- 벡터 데이터베이스 기반 RAG 구현
- RAG 검색·생성 성능 개선 및 평가
- Tool, Agent, SQL, MCP 등 실용 기능 활용
- GPT Builder와 Action을 이용한 커스텀 GPT 개발

### 과정 구성
- **Day 1:** LangChain 컴포넌트, 자동화·챗봇, 벡터 스토어와 RAG
- **Day 2:** Advanced RAG, RAG 평가, LangChain 실용 예제, GPT Builder

---

## 2. LangChain 기본 구조

### LangChain의 특징
LangChain은 LLM 애플리케이션 개발에 필요한 기능을 모듈화한 **“LLM Application Toolbox”**다.

- 모델, 프롬프트, 체인 연결
- 데이터베이스 및 메모리 연동
- Tool과 Agent 구성
- 높은 추상화를 통한 빠른 구현
- OpenAI, Anthropic, Google, Hugging Face, Ollama 등 지원

Provider별 라이브러리로 `langchain_openai`, `langchain_anthropic`, `langchain_google_genai`, `langchain_huggingface`, `langchain_ollama` 등이 사용된다.

### LangChain 1.0 변화
2025년 10월 LangChain 1.0에서 라이브러리가 재편되었다.

- `langchain`: 주로 LLM Agent 

---

## 2. Map Reduce 방식

Map Reduce 방식은 문서를 여러 청크로 나누어 처리합니다.

1. Map 단계: 각 청크를 개별적으로 요약
2. Reduce 단계: 개별 요약들을 합쳐서 최종 요약 생성

### 장점
- 매우 긴 문서도 처리 가능
- 병렬 처리로 속도 향상 가능

### 단점
- 청크 간의 맥락이 손실될 수 있음
- API 호출 횟수가 증가

```
[청크1] → [요약1] ─┐
[청크2] → [요약2] ─┼→ [최종 요약]
[청크3] → [요약3] ─┘
```

In [ ]:
# Map 단계 프롬프트 (개별 청크 요약)
map_prompt = ChatPromptTemplate([
    ("system", """당신은 문서 요약 전문가입니다.
주어진 문서의 일부분을 읽고 핵심 내용을 요약해주세요.
이 요약은 나중에 다른 부분의 요약과 합쳐져 전체 요약이 됩니다."""),
    ("human", """다음 문서 일부를 요약해주세요:

---
{text}
---

핵심 내용을 간결하게 요약해주세요.""")
])

# Reduce 단계 프롬프트 (요약들을 합쳐서 최종 요약)
reduce_prompt = ChatPromptTemplate([
    ("system", """당신은 문서 요약 전문가입니다.
여러 부분의 요약들을 받아서 하나의 일관된 최종 요약을 작성해주세요.
중복되는 내용은 통합하고, 전체적인 흐름이 자연스럽게 연결되도록 해주세요."""),
    ("human", """다음은 문서의 여러 부분에 대한 요약들입니다:

---
{summaries}
---

위 요약들을 통합하여 전체 문서의 최종 요약을 작성해주세요.""")
])

In [ ]:
# Map 체인과 Reduce 체인
map_chain = map_prompt | llm | StrOutputParser()
reduce_chain = reduce_prompt | llm | StrOutputParser()

In [ ]:
# Map Reduce 실행
print("Map Reduce 방식으로 요약 중...")
start_time = time.time()

# Map 단계: 각 청크 요약
print(f"\n[Map 단계] {len(chunks)}개 청크 요약 중...")
chunk_summaries = map_chain.batch([{"text": c.page_content} for c in chunks])

print('개별 요약 처리 완료!')

# Reduce 단계: 요약들 통합
print("\n[Reduce 단계] 요약 통합 중...")
combined_summaries = "\n\n".join(chunk_summaries)
map_reduce_summary = reduce_chain.invoke({"summaries": combined_summaries})

elapsed_time = time.time() - start_time
print(f"\n완료! (소요 시간: {elapsed_time:.2f}초)\n")
print("="*60)
print("[Map Reduce 요약 결과]")
print("="*60)
print(map_reduce_summary)

Map Reduce 방식으로 요약 중...

[Map 단계] 11개 청크 요약 중...
개별 요약 처리 완료!

[Reduce 단계] 요약 통합 중...

완료! (소요 시간: 43.56초)

[Map Reduce 요약 결과]
## 최종 요약

이 문서는 2026년 삼성SDS **「RAG 개발 심화」 과정**의 교육 자료로, 변형호 강사가 진행한다. 과정의 목표는 LLM과 RAG의 핵심 원리를 이해하고, LangChain을 활용한 애플리케이션 개발부터 검색 성능 개선·평가, Tool·Agent, GPT Builder와 Actions까지 실습하는 것이다. 1일 차에는 LangChain 컴포넌트, 자동화·챗봇, 벡터스토어와 RAG를 다루고, 2일 차에는 RAG 고도화 및 평가, 실용 예제와 GPT Builder를 학습한다.

### 1. LangChain 기반 LLM 애플리케이션 개발

LangChain은 모델, 프롬프트, 체인, 데이터베이스, 메모리, 도구, 에이전트를 모듈식으로 연결하는 고수준 프레임워크다. OpenAI, Google, Hugging Face, Ollama 등 다양한 모델 제공자와 연동할 수 있으며, `init_chat_model()`과 System·Human·AI 메시지, 프롬프트 템플릿을 이용해 채팅 모델을 통합적으로 구성한다.

핵심 실행 단위는 `Runnable`이며 `invoke/ainvoke`, `batch/abatch`, `stream`을 통해 동기·비동기 실행, 병렬 처리와 스트리밍 출력을 지원한다. `RunnablePassthrough`, `RunnableParallel`, `.assign()`을 사용하면 데이터를 전달하거나 여러 작업을 병렬로 실행·결합할 수 있고, 이러한 체인 구조는 LangGraph의 그래프형 워크플로로 확장될 수 있다. `with_structured_output()`을 활용하면 LLM의 출력을 JSON, Pydantic, 날짜·시간 등 구조화된 형태로 받아 후속 처리에 바로 사용할 수 있다.

LangCh

---

## 3. Refine 방식

Refine 방식은 청크를 순차적으로 처리하며 요약을 점진적으로 개선합니다.

1. 첫 번째 청크로 초기 요약 생성
2. 다음 청크와 현재 요약을 함께 보고 요약 개선
3. 모든 청크를 처리할 때까지 반복

### 장점
- 문서의 맥락을 유지하며 요약
- 순차적으로 정보가 누적됨

### 단점
- 순차 처리로 시간이 오래 걸림
- 앞부분 내용이 뒷부분에 의해 희석될 수 있음

```
[청크1] → [요약v1]
              ↓
[청크2] + [요약v1] → [요약v2]
                        ↓
[청크3] + [요약v2] → [최종 요약]
```

In [ ]:
# 초기 요약 프롬프트 (첫 번째 청크용)
initial_prompt = ChatPromptTemplate([
    ("system", """당신은 문서 요약 전문가입니다.
문서의 첫 부분을 읽고 초기 요약을 작성해주세요.
이 요약은 이후 문서의 다른 부분을 읽으면서 점진적으로 개선될 것입니다."""),
    ("human", """다음은 문서의 첫 부분입니다:

---
{text}
---

이 내용을 바탕으로 초기 요약을 작성해주세요.""")
])

# 개선 프롬프트 (후속 청크용)
refine_prompt = ChatPromptTemplate([
    ("system", """당신은 문서 요약 전문가입니다.
기존 요약과 새로운 문서 부분을 함께 보고, 요약을 개선해주세요.
새로운 정보가 있다면 추가하고, 기존 내용과 통합하여 일관된 요약을 만들어주세요."""),
    ("human", """현재까지의 요약:
{current_summary}

---

새로운 문서 부분:
{text}

---

위의 새로운 내용을 반영하여 요약을 개선해주세요.""")
])

In [ ]:
# Refine 체인들
initial_chain = initial_prompt | llm | StrOutputParser()
refine_chain = refine_prompt | llm | StrOutputParser()

In [ ]:
# Refine 실행
print("Refine 방식으로 요약 중...")
start_time = time.time()

# 첫 번째 청크로 초기 요약 생성
print(f"\n[초기 요약] 청크 1/{len(chunks)} 처리 중...")
current_summary = initial_chain.invoke({"text": chunks[0].page_content})
print(f"  청크 1/{len(chunks)} 완료")

# 나머지 청크들로 요약 개선
for i, chunk in enumerate(chunks[1:], start=2):
    print(f"  청크 {i}/{len(chunks)} 처리 중...")
    current_summary = refine_chain.invoke({
        "current_summary": current_summary,
        "text": chunk.page_content
    })
    print(f"  청크 {i}/{len(chunks)} 완료")

refine_summary = current_summary

elapsed_time = time.time() - start_time
print(f"\n완료! (소요 시간: {elapsed_time:.2f}초)\n")
print("="*60)
print("[Refine 요약 결과]")
print("="*60)
print(refine_summary)

Refine 방식으로 요약 중...

[초기 요약] 청크 1/11 처리 중...
  청크 1/11 완료
  청크 2/11 처리 중...
  청크 2/11 완료
  청크 3/11 처리 중...
  청크 3/11 완료
  청크 4/11 처리 중...
  청크 4/11 완료
  청크 5/11 처리 중...
  청크 5/11 완료
  청크 6/11 처리 중...
  청크 6/11 완료
  청크 7/11 처리 중...
  청크 7/11 완료
  청크 8/11 처리 중...
  청크 8/11 완료
  청크 9/11 처리 중...


---

## 세 가지 방식 비교

| 방식 | 특징 | 적합한 상황 |
|------|------|-------------|
| Stuff | 전체를 한 번에 처리 | 짧은 문서, 빠른 결과 필요시 |
| Map Reduce | 병렬 처리 후 통합 | 긴 문서, 병렬 처리 가능시 |
| Refine | 순차적 개선 | 맥락 유지가 중요한 경우 |

## [실습] 임의의 PDF 다운로드하여 요약하기

arxiv 등의 페이지에서 PDF를 다운로드하여 업로드하고,   
Stuff/Map-Reduce/Refine 등의 방법을 이용해 전체 PDF를 요약하세요.